<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0;">Lab 01: MAF Environment Setup &amp; Validation</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">Install the Microsoft Agent Framework SDK and validate your environment for building custom hosted agents</p>
</div>

**What We Learn:** How to install the Microsoft Agent Framework SDK, understand the hosted agent architecture, and validate your environment for building custom agents that serve the Contoso Travel scenario.

---

## 🧳 The Contoso Travel Story

Contoso Travel has already built prompted agents for their Flight, Hotel, and Car Rental services. But prompted agents run entirely within Foundry — there's no way to embed custom business logic, integrate with internal APIs, or run specialized ML models inside the agent process itself.

- **The Problem:** The team needs agents that can run custom Python code — complex pricing algorithms, loyalty tier calculations, and direct database queries — inside the agent process for lower latency and more control.
- **This Lab Solves:** Setting up the Microsoft Agent Framework (MAF) SDK so the team can build custom agents that run their own Python code while still being managed by Foundry.

## 1. Install MAF Dependencies

The Microsoft Agent Framework SDK (`azure-ai-agentserver-agentframework`) provides the base classes and hosting adapter needed to build custom hosted agents.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Tip:</b> This installs the same <code>azure-ai-projects</code> and <code>azure-identity</code> packages used in the prompted agents labs, plus the MAF-specific SDK.
</div>

---

In [ ]:
# Install the Microsoft Agent Framework SDK and dependencies
%pip install azure-ai-agentserver-agentframework==1.0.0b16 azure-ai-projects>=2.0.0 azure-identity python-dotenv --quiet

## 2. Load Environment & Connect to Foundry

We reuse the same `.env` file from the prompted agents labs — the Foundry project endpoint and model deployment are shared across all tracks.

---

In [ ]:
# Load environment and connect to the Foundry project
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load shared environment variables
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

print(f"✅ Connected to Foundry project")
print(f"   Model: {model_name}")

## 3. Understand the MAF Architecture

The Microsoft Agent Framework provides a layered class hierarchy for building hosted agents:

- **`BaseAgent`** — The low-level base class. You implement `run()` and optionally `run_stream()` to define your agent's behavior from scratch.
- **`Agent`** — Higher-level class with built-in LLM integration and tool support. Handles model calls and tool dispatch so you can focus on business logic.
- **`ChatAgent`** — Extends `Agent` with multi-turn conversation management, maintaining chat history across interactions.
- **`from_agent_framework()`** — The hosting adapter that wraps your agent class and serves it via Foundry's Responses API. This is what makes your custom code accessible as a hosted agent.

### How the pieces connect

```
┌──────────────────────┐     ┌─────────────────────────┐     ┌───────────────────────┐     ┌────────────────┐
│  Your Code           │     │  from_agent_framework() │     │  Foundry Agent        │     │  Responses API │
│  (BaseAgent / Agent) │ ──▶ │  (hosting adapter)      │ ──▶ │  Service              │ ──▶ │  (clients)     │
└──────────────────────┘     └─────────────────────────┘     └───────────────────────┘     └────────────────┘
```

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Tip:</b> In these labs we run agents locally for development. The <code>from_agent_framework()</code> adapter is what enables the same code to run hosted in Foundry when deployed.
</div>

---

## 4. Understand agent.yaml

When deploying a hosted agent to Foundry Agent Service, you provide an `agent.yaml` manifest that describes the agent:

```yaml
kind: hosted
name: contoso-travel-agent
model: gpt-4.1-mini
description: Contoso Travel concierge agent built with MAF
```

Key fields:
- **`kind: hosted`** — tells Foundry this is a custom-code agent (not a prompted agent)
- **`name`** — the agent identifier used in API calls
- **`model`** — the default model deployment the agent will use
- **`description`** — human-readable description shown in Foundry portal

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #ff9800; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>⚠️ Note:</b> For local development (these labs), we don't need <code>agent.yaml</code> — it's used when deploying to Foundry Agent Service. See <code>src/contoso-travel-maf/</code> for a complete deployable agent including <code>agent.yaml</code>, <code>agent.py</code>, and the <code>deploy.sh</code> script.
</div>

---

## 5. Prompted vs Hosted — Side-by-Side

Understanding when to use each approach is key to the Contoso Travel architecture:

| Aspect | Prompted Agents | Hosted Agents (MAF) |
|--------|----------------|---------------------|
| Code | No custom code — instructions + tools via SDK | Your Python code in a container |
| Tools | FunctionTool definitions, Foundry executes | Python functions run in-process |
| Latency | Extra round-trip for tool calls | Tools execute locally (faster) |
| Control | Limited to SDK capabilities | Full Python — custom logic, ML models, databases |
| Deployment | No deployment needed | `azd agent deploy` to Foundry Agent Service |
| Tracing | `AIProjectInstrumentor` | Auto-instrumented by hosting adapter |

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Tip:</b> Prompted agents are great for simple tool-calling patterns. Choose hosted agents when you need custom Python logic, lower latency tool execution, or integration with internal systems.
</div>

---

## 6. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #4caf50; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>✅ What we accomplished:</b><br>
  • Installed the Microsoft Agent Framework SDK (<code>azure-ai-agentserver-agentframework</code>)<br>
  • Connected to the Foundry project using the shared <code>.env</code> configuration<br>
  • Understood the MAF class hierarchy: <code>BaseAgent</code> → <code>Agent</code> → <code>ChatAgent</code><br>
  • Learned how <code>from_agent_framework()</code> bridges custom code to Foundry's Responses API<br>
  • Compared prompted vs hosted agent trade-offs for Contoso Travel
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #ff9800; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>👉 Next — Lab 02:</b> Build your first hosted agent using the <code>Agent</code> class with custom Python tool functions for Contoso Travel's flight search.
</div>

---

In [ ]:
# Close client connections and release resources
project_client.close()
credential.close()
print("✅ Clients closed. Setup complete!")